# QNAM: Noncrossing additive quantile regression

QNAM predicts several conditional quantiles while constraining every intercept and term contribution to be nondecreasing across ordered quantile levels.


## Model


For $\tau_1<\cdots<\tau_K$,

$$
q_{\tau_k}(x)=\beta_{0k}+\sum_j f_{jk}(x_j),
\qquad
q_{\tau_1}(x)\le\cdots\le q_{\tau_K}(x).
$$

Positive transformed increments enforce the ordering at every additive component.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_method` and `categorical_method` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import QNAMLSS


quantiles = [0.1, 0.5, 0.9]
model = QNAMLSS(
    layer_sizes=[32, 16],
    monotone_transform="softplus",
    min_increment=0.0,
    dropout=0.0,
    distributional_kwargs={"quantiles": quantiles},
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


## Model-specific controls

QNAM is intentionally distributional-only. Set quantile levels with the estimator's `distributional_kwargs`; `predict` returns one ordered column per level.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train, y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predicted_quantiles = model.predict(X_test)
    assert np.all(np.diff(predicted_quantiles, axis=1) >= -1e-7)
    score = model.score(X_test, y_test)
    display(model.evaluate(X_test, y_test))
    model.predict_components(X_test).validate_additive_reconstruction()


## Task variants and limits

QNAM exposes `QNAMLSS` only because its output contract is an ordered quantile distribution.
